# ⌨️ Cookbook: the two-party flow without Python

Keys, encryption, and decryption as shell commands — the shape this takes
when the controller's side is an ops team, not a data science team.

In [ ]:
%%time
%pip install -q git+https://github.com/PDPG-lab/pypdpg

## Controller: keys and data

In [ ]:
!pdpg keygen -o keys

In [ ]:
# a CSV like any export from any system
import numpy as np, csv
rng = np.random.default_rng(7)
with open("applicants.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["income", "debt", "age"])
    w.writerows(np.column_stack([
        rng.normal(58_000, 18_000, 50).clip(18_000, 150_000),
        rng.normal(22_000, 12_000, 50).clip(0, 90_000),
        rng.uniform(21, 70, 50),
    ]).round(2).tolist())
print(open("applicants.csv").read(120), "...")

In [ ]:
!pdpg encrypt applicants.csv -c keys/controller.key -o data.enc

## The interceptor's view

`data.enc` gets stolen in transit. What does the thief learn? `inspect`
needs no key — it shows *everything* an outsider can extract from the file:

In [ ]:
!pdpg inspect data.enc

In [ ]:
!pdpg inspect keys/processor.ctx

## The processor cannot decrypt — by construction

In [ ]:
!pdpg decrypt data.enc -c keys/processor.ctx || echo "(exit code $?)"

## The controller can

In [ ]:
!pdpg decrypt data.enc -c keys/controller.key -o back.csv
!head -4 back.csv

Header row and shape survive the roundtrip; values were ciphertext the
whole way. In between those two commands, the processor computes — that
part is Python (`np.load("data.enc")` and friends), see
[getting_started](https://colab.research.google.com/github/PDPG-lab/pypdpg/blob/main/demo/getting_started.ipynb).

<sub>A [PDPG-lab](https://pdpglab.xyz) project. Current backend:
[TenSEAL](https://github.com/OpenMined/TenSEAL) (CKKS). More recipes in
[demo/cookbook](https://github.com/PDPG-lab/pypdpg/tree/main/demo/cookbook).</sub>